## Concept focus — Exception hierarchy design

Advanced error handling is about shaping failures so callers can recover intentionally. A clear exception hierarchy makes programs easier to debug, test, and operate because each failure communicates both type and meaning.

```text
Exception
└── AppError
    ├── ConfigError
    ├── ValidationError
    └── NetworkError

callers can catch broad or specific failures deliberately
```

### How to think about it
Design exceptions like an API. A caller should be able to decide whether to retry, report, ignore, or crash based on the class of failure, not fragile string matching against the message text.

### Visual references and further study
- [Python docs — errors and exceptions](https://docs.python.org/3/tutorial/errors.html)
- [Python exception hierarchy](https://docs.python.org/3/library/exceptions.html)
- [PEP 654 — Exception Groups](https://peps.python.org/pep-0654/)
- [Real Python — raising exceptions](https://realpython.com/python-raise-exception/)

---

# Module 16 — Error Handling and Robustness

## Exercise 16.1 — Twelve predictions about control flow

Predict the exact output of each before running.
Run:  python ex01_hierarchy.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. The exception hierarchy

```text
BaseException
├── SystemExit              raised by sys.exit() and SystemExit
├── KeyboardInterrupt       Ctrl-C
├── GeneratorExit           generator.close()
└── Exception               <- everything you should ever catch
    ├── ArithmeticError → ZeroDivisionError, OverflowError
    ├── LookupError      → IndexError, KeyError
    ├── OSError          → FileNotFoundError, PermissionError,
    │                        ConnectionError, TimeoutError, IsADirectoryError
    ├── ValueError       → UnicodeDecodeError
    ├── TypeError, AttributeError, NameError, ImportError
    ├── RuntimeError     → RecursionError, NotImplementedError
    └── StopIteration, StopAsyncIteration
```


**The three under `BaseException` and not `Exception` are there deliberately.**
`except Exception:` does not catch `KeyboardInterrupt` or `SystemExit`, which
is exactly what you want — Ctrl-C should stop your program even inside a retry
loop, and `sys.exit()` should exit.

Which is why:

```text
except:                     # NEVER. Catches Ctrl-C and SystemExit.
except BaseException:       # almost never. Same problem, more explicit.
except Exception:           # the outermost handler of a long-running process
except (OSError, ValueError):   # what you should usually write
```


**Catch the narrowest exception you can actually handle.** "Handle" means: you
can do something about it. Logging and re-raising is not handling; it is
duplicating what the traceback already says.

---

## Concept 2. `try/except/else/finally`, precisely

In [ ]:
try:
    result = risky()
except ValueError as exc:
    handle(exc)
else:
    use(result)          # runs ONLY if no exception was raised
finally:
    cleanup()            # runs ALWAYS: success, exception, return, break

**`else` exists to keep the `try` block minimal.** Compare:

In [ ]:
try:
    value = d[key]
    process(value)          # if THIS raises KeyError, it is caught by mistake
except KeyError:
    ...

try:
    value = d[key]
except KeyError:
    ...
else:
    process(value)          # its KeyErrors are NOT caught here

The first version silently swallows an unrelated `KeyError` from deep inside
`process`, and you get the "key not found" branch for a completely different
reason. **Put exactly the line that can raise inside the `try`.**

**`finally` runs even on `return`**, which produces one genuine surprise:

In [ ]:
def f():
    try:
        return "from try"
    finally:
        return "from finally"     # this WINS, and discards the exception too

A `return` in `finally` swallows any in-flight exception. Never do it; linters
flag it (`ruff` rule `B012`).

---

## Concept 4. Designing your own exceptions

In [ ]:
class AppError(Exception):
    """Base for everything this application raises deliberately."""

class ValidationError(AppError):
    def __init__(self, field: str, value: object, reason: str) -> None:
        super().__init__(f"{field}={value!r}: {reason}")
        self.field = field
        self.value = value
        self.reason = reason

**Four rules.**

**1. One base class per package.** A caller can then write
`except AppError:` and catch everything you raise deliberately, while still
seeing bugs (a `TypeError` from your own code) propagate.

**2. Carry data, not just a string.** An exception with only a message forces
every handler to parse English to react. Carrying `field`, `value` and `reason`
lets a caller decide, and lets a web layer render JSON.

**3. The message must identify the input.** Compare:

```text
ValueError: invalid input
ValueError: line 4210: expected 3 fields, got 2: 'grace,45'
```


The second one costs eight extra characters to write and saves an hour. **Use
`!r`** — it shows quotes and whitespace, which is exactly what matters when the
bug is a trailing space or an empty string.

**4. Do not inherit from `BaseException`.** Ever.

---

## Concept 8. Retries, timeouts, and backoff

In [ ]:
def with_retry(fn, attempts=3, base_delay=0.1, max_delay=10.0):
    for attempt in range(attempts):
        try:
            return fn()
        except (ConnectionError, TimeoutError) as exc:
            if attempt == attempts - 1:
                raise
            delay = min(base_delay * 2**attempt, max_delay)
            delay *= 0.5 + random.random()          # JITTER -- see below
            time.sleep(delay)

**Four rules.**

**Only retry transient failures.** A `ValidationError` will fail identically
every time; retrying it wastes time and hides the real problem. Retry
`ConnectionError`, `TimeoutError`, and HTTP 429/502/503/504. Never retry 400,
401, 403, 404, or 422.

**Only retry idempotent operations.** If `charge_card()` times out you do not
know whether the charge happened. Retrying may charge twice. The fix is an
**idempotency key**: the caller generates a unique ID, the server records it,
and a repeat with the same key returns the original result. Module 33.

**Exponential backoff with jitter.** Without jitter, a thousand clients that
failed together retry together, forever — the thundering herd. Randomising the
delay spreads them out. This is not a refinement; it is the difference between
recovery and a self-sustaining outage.

**Always set a timeout.** Every network call, every lock acquisition, every
queue `get`. A call with no timeout is a hang waiting for a bad day, and the
default for most libraries is *no timeout*.

In [ ]:
httpx.get(url, timeout=5.0)       # not httpx.get(url)
lock.acquire(timeout=1.0)
queue.get(timeout=30)

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: The exception hierarchy
- Section 2: `try/except/else/finally`, precisely
- Section 3: Exception chaining
- Section 4: Designing your own exceptions
- Section 5: EAFP and LBYL
- Section 6: `ExceptionGroup` and `except*` (3.11+)
- Section 7: Logging
- Section 8: Retries, timeouts, and backoff
- Section 9: What never to do

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

---

## `q01`

_q01_

In [ ]:
def q01() -> str:
    # PREDICTION:
    try:
        return "try"
    finally:
        print("    q01 finally ran")

---

## `q02`

_q02_

In [ ]:
def q02() -> str:
    # PREDICTION: which value is returned, and what happens to the exception?
    try:
        raise ValueError("boom")
    finally:
        return "finally wins"        # noqa: B012

---

## `q03`

_q03_

In [ ]:
def q03() -> None:
    # PREDICTION: does the else clause run?
    for label, value in [("present", {"k": 1}), ("absent", {})]:
        try:
            v = value["k"]
        except KeyError:
            print(f"    q03 {label}: caught")
        else:
            print(f"    q03 {label}: else ran, v={v}")
        finally:
            print(f"    q03 {label}: finally")

---

## `q04`

_q04_

In [ ]:
def q04() -> None:
    # PREDICTION: which handler catches, and why is that a bug?
    def process(v: int) -> None:
        raise KeyError("something ELSE went wrong deep inside process")

    data = {"k": 1}
    try:
        value = data["k"]
        process(value)
    except KeyError as exc:
        print(f"    q04 caught: {exc}")
        print("    q04 -> and the code now thinks the KEY was missing")

---

## `q05`

_q05_

In [ ]:
def q05() -> None:
    # PREDICTION: is the ValueError visible in the traceback? What connects them?
    try:
        try:
            raise ValueError("original")
        except ValueError:
            raise RuntimeError("replacement")
    except RuntimeError as exc:
        print(f"    q05 __context__: {exc.__context__}")
        print(f"    q05 __cause__:   {exc.__cause__}")

---

## `q06`

_q06_

In [ ]:
def q06() -> None:
    # PREDICTION: same question, with `from`.
    try:
        try:
            raise ValueError("original")
        except ValueError as inner:
            raise RuntimeError("replacement") from inner
    except RuntimeError as exc:
        print(f"    q06 __context__: {exc.__context__}")
        print(f"    q06 __cause__:   {exc.__cause__}")

---

## `q07`

_q07_

In [ ]:
def q07() -> None:
    # PREDICTION: what does `from None` leave behind?
    try:
        try:
            raise ValueError("original")
        except ValueError:
            raise RuntimeError("replacement") from None
    except RuntimeError as exc:
        print(f"    q07 __context__: {exc.__context__}")
        print(f"    q07 __cause__:   {exc.__cause__}")
        print(f"    q07 suppress:    {exc.__suppress_context__}")

---

## `q08`

_q08_

In [ ]:
def q08() -> None:
    # PREDICTION: is `exc` still bound after the except block?
    try:
        raise ValueError("boom")
    except ValueError as exc:
        message = str(exc)
    try:
        print(f"    q08 exc is {exc}")      # type: ignore[possibly-undefined]
    except NameError:
        print(f"    q08 NameError -- exc was deleted. message={message!r}")

---

## `q09`

_q09_

In [ ]:
def q09() -> None:
    # PREDICTION: which of these does `except Exception` catch?
    import sys
    for exc_cls in (ValueError, KeyboardInterrupt, SystemExit, GeneratorExit):
        try:
            raise exc_cls("x")
        except Exception:
            print(f"    q09 {exc_cls.__name__:<18} caught by except Exception")
        except BaseException:
            print(f"    q09 {exc_cls.__name__:<18} NOT caught -- BaseException")

---

## `q10`

_q10_

In [ ]:
def q10() -> None:
    # PREDICTION: what order do the finallys run in?
    def inner() -> None:
        try:
            raise ValueError("deep")
        finally:
            print("    q10 inner finally")

    def middle() -> None:
        try:
            inner()
        finally:
            print("    q10 middle finally")

    try:
        middle()
    except ValueError:
        print("    q10 caught at the top")

---

## `q11`

_q11_

In [ ]:
def q11() -> None:
    # PREDICTION: what happens when the finally itself raises?
    try:
        try:
            raise ValueError("original")
        finally:
            raise RuntimeError("from finally")
    except Exception as exc:
        print(f"    q11 surfaced: {type(exc).__name__}: {exc}")
        print(f"    q11 the original is at __context__: {exc.__context__}")

---

## `q12`

_q12_

In [ ]:
def q12() -> None:
    # PREDICTION: how many times does the loop body run?
    attempts = 0
    for i in range(3):
        try:
            attempts += 1
            raise ConnectionError("transient")
        except ConnectionError:
            continue
        finally:
            print(f"    q12 finally on iteration {i}")
    print(f"    q12 attempts={attempts}")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    print("q01:", q01())
    print("q02:", q02())
    for fn in [q03, q04, q05, q06, q07, q08, q09, q10, q11, q12]:
        fn()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.